# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/surajprai2111-ui/ML-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I built a simple score to decide which content pages are worth checking first. Pages with more search visibility can be worth reviewing because a change to them could affect more search traffic. I also considered their current search position, how long ago they were updated, and their content length.

The score is based on these four components:

- **Visibility:** Pages with more impressions receive a higher score.
- **Freshness:** Pages with older updates receive a higher score, although the data did not show a consistent relationship between update age and decline.
- **Position opportunity:** Pages ranking between positions 1 and 20 receive a score based on their current position.
- **Content depth:** Shorter pages receive a small additional score when word count is available.

I use `declining_with_visibility` as a reason code to flag pages that show a downward trend and have at least 100 impressions in the 90-day period. This reason code helps explain why a page deserves attention, but it does not directly increase the numerical score.

The suggested action is `refresh_review`, meaning the page should be checked by a person before any change is made. A high score is not proof that refreshing the page will improve its performance.

In [22]:
import pandas as pd

# Load the starter dataset
url = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(url)

print("Dataset shape:", df.shape)

# Keep the same population used for the baseline
base = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].drop_duplicates("content_id").copy()

# Use the observed trend only to check the rule.
# Do not use this label to calculate the baseline score.
base["is_declining"] = (
    base["trend_direction"] == "down"
).astype(int)

# Convert update age to numeric values
base["days_since_last_update"] = pd.to_numeric(
    base["days_since_last_update"],
    errors="coerce"
)

print("\nBaseline population:", len(base))
print("Declining pages:", base["is_declining"].sum())
print(
    "Overall decline rate:",
    round(base["is_declining"].mean() * 100, 2),
    "%"
)

# --------------------------------------------------
# Check 1: Does update age show a consistent pattern?
# Include pages updated less than 90 days ago.
# --------------------------------------------------

age_bins = [0, 90, 180, 365, 730, float("inf")]

age_labels = [
    "0-3 months",
    "3-6 months",
    "6-12 months",
    "12-24 months",
    "24+ months"
]

base["update_age_group"] = pd.cut(
    base["days_since_last_update"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

# Report missing update dates separately
base["update_age_group"] = (
    base["update_age_group"]
    .cat.add_categories(["Missing update date"])
    .fillna("Missing update date")
)

age_check = (
    base.groupby("update_age_group", observed=False)
    .agg(
        pages=("content_id", "count"),
        declining_pages=("is_declining", "sum"),
        decline_rate=("is_declining", "mean")
    )
)

age_check["decline_rate"] = (
    age_check["decline_rate"] * 100
).round(2)

print("\nDecline rate by days since last update:")
print(age_check)

# --------------------------------------------------
# Check 2: Which impression levels are meaningful?
# --------------------------------------------------

impression_bins = [
    0, 100, 250, 500, 1000, 5000, float("inf")
]

impression_labels = [
    "<100",
    "100-249",
    "250-499",
    "500-999",
    "1,000-4,999",
    "5,000+"
]

base["impression_group"] = pd.cut(
    base["impressions_90d"],
    bins=impression_bins,
    labels=impression_labels,
    right=False
)

impression_check = (
    base.groupby("impression_group", observed=False)
    .agg(
        pages=("content_id", "count"),
        declining_pages=("is_declining", "sum"),
        decline_rate=("is_declining", "mean")
    )
)

impression_check["decline_rate"] = (
    impression_check["decline_rate"] * 100
).round(2)

print("\nDecline rate by impressions:")
print(impression_check)

# --------------------------------------------------
# Check 3: Count missing and invalid update-age values
# --------------------------------------------------

missing_updates = base["days_since_last_update"].isna().sum()
negative_updates = (
    base["days_since_last_update"] < 0
).sum()

print("\nUpdate-date quality check:")
print("Missing update-age values:", missing_updates)
print("Negative update-age values:", negative_updates)

Dataset shape: (30000, 44)

Baseline population: 30000
Declining pages: 16262
Overall decline rate: 54.21 %

Decline rate by days since last update:
                     pages  declining_pages  decline_rate
update_age_group                                         
0-3 months           20655            10576         51.20
3-6 months            9171             5604         61.11
6-12 months            169               79         46.75
12-24 months             5                3         60.00
24+ months               0                0           NaN
Missing update date      0                0           NaN

Decline rate by impressions:
                  pages  declining_pages  decline_rate
impression_group                                      
<100               7994             3110         38.90
100-249            2620             1605         61.26
250-499            2660             1586         59.62
500-999            3214             1930         60.05
1,000-4,999        7361    

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Load the starter dataset
# --------------------------------------------------

url = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(url)

print("Dataset shape:", df.shape)


# --------------------------------------------------
# 2. Prepare the baseline population
# --------------------------------------------------

required_columns = [
    "content_id",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "trend_direction",
    "days_since_last_update",
    "content_age_days",
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Keep pages with search visibility and at least 90 days of content age.
base = df.loc[
    (df["impressions_90d"] > 0)
    & (df["content_age_days"] >= 90)
].copy()

# Keep one row per content item.
base = base.drop_duplicates(subset="content_id").copy()

# Convert numeric columns safely.
numeric_columns = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "days_since_last_update",
    "content_age_days",
]

for col in numeric_columns:
    base[col] = pd.to_numeric(base[col], errors="coerce")

print("Baseline population:", len(base))
print("Unique content IDs:", base["content_id"].nunique())


# --------------------------------------------------
# 3. Create transparent score components
# --------------------------------------------------

# Visibility:
# More impressions indicate that a page reaches more search users.
# Log scaling prevents extremely high impression counts from dominating.
base["visibility_score"] = (
    np.log1p(base["impressions_90d"])
    / np.log1p(base["impressions_90d"].max())
) * 100

# Freshness risk:
# Older updates receive a higher score.
# Missing update dates receive 0 rather than being assumed stale.
base["freshness_risk_score"] = (
    base["days_since_last_update"].clip(lower=0, upper=365) / 365
) * 100

base["freshness_risk_score"] = (
    base["freshness_risk_score"].fillna(0)
)

# Position opportunity:
# Pages ranking between positions 1 and 20 may have room to improve.
# Positions closer to 20 receive a higher opportunity score.
position = base["avg_position"]

base["position_opportunity_score"] = np.where(
    position.between(1, 20),
    ((position - 1) / 19) * 100,
    0,
)

# Depth gap:
# Shorter pages receive a higher review score.
# Missing word counts receive 0 because missing does not mean short.
base["depth_gap_score"] = np.select(
    [
        base["word_count"].between(1, 599),
        base["word_count"].between(600, 1199),
    ],
    [
        100,
        50,
    ],
    default=0,
)


# --------------------------------------------------
# 4. Calculate the baseline action score
# --------------------------------------------------

base["baseline_action_score"] = (
    0.40 * base["visibility_score"]
    + 0.30 * base["freshness_risk_score"]
    + 0.25 * base["position_opportunity_score"]
    + 0.05 * base["depth_gap_score"]
)

base["baseline_action_score"] = (
    base["baseline_action_score"].round(2)
)


# --------------------------------------------------
# 5. Assign reason codes
# --------------------------------------------------

reason_codes = []

for _, row in base.iterrows():
    reasons = []

    # Meaningful search visibility
    if row["impressions_90d"] >= 500:
        reasons.append("meaningful_visibility")

    # Page is relatively old and has meaningful visibility
    if (
        pd.notna(row["days_since_last_update"])
        and row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 500
    ):
        reasons.append("stale_visible_page")

    # Declining trend with some search demand
    if (
        row["trend_direction"] == "down"
        and row["impressions_90d"] >= 100
    ):
        reasons.append("declining_with_visibility")

    # Page ranks in the first 20 positions
    if (
        pd.notna(row["avg_position"])
        and 0 < row["avg_position"] <= 20
    ):
        reasons.append("position_opportunity")

    # CTR is low despite meaningful visibility and a reasonable position
    if (
        row["impressions_90d"] >= 500
        and pd.notna(row["avg_position"])
        and 0 < row["avg_position"] <= 20
        and pd.notna(row["ctr"])
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    # Short content with meaningful visibility
    if (
        pd.notna(row["word_count"])
        and 0 < row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    # Keep every row interpretable, even if no specific rule matched.
    if not reasons:
        reasons.append("general_review")

    reason_codes.append("|".join(reasons))

base["reason_codes"] = reason_codes


# --------------------------------------------------
# 6. Assign the suggested action
# --------------------------------------------------

base["action"] = "refresh_review"


# --------------------------------------------------
# 7. Rank all pages
# --------------------------------------------------

output = base.sort_values(
    by=["baseline_action_score", "impressions_90d"],
    ascending=[False, False],
).copy()

output["rank"] = np.arange(1, len(output) + 1)

# Keep useful review fields in the CSV.
output_columns = [
    "rank",
    "content_id",
    "baseline_action_score",
    "reason_codes",
    "action",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "days_since_last_update",
    "content_age_days",
]

output_columns = [
    col for col in output_columns if col in output.columns
]

output = output[output_columns]


# --------------------------------------------------
# 8. Save the required CSV
# --------------------------------------------------

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(output_path, index=False)


# --------------------------------------------------
# 9. Print checks and the Top 20 review queue
# --------------------------------------------------

print("\nRows in ranked queue:", len(output))
print("Unique content IDs:", output["content_id"].nunique())
print(
    "Score range:",
    output["baseline_action_score"].min(),
    "to",
    output["baseline_action_score"].max(),
)

print("\nTop 20 pages:")
print(output.head(20).to_string(index=False))

print("\nReason-code counts:")
print(
    output["reason_codes"]
    .str.split("|")
    .explode()
    .value_counts()
)

print("\nSaved to:", output_path)

Dataset shape: (30000, 44)
Baseline population: 30000
Unique content IDs: 30000

Rows in ranked queue: 30000
Unique content IDs: 30000
Score range: 2.19 to 74.08

Top 20 pages:
 rank           content_id  baseline_action_score                                                                                                 reason_codes         action  impressions_90d  avg_position  ctr  word_count  days_since_last_update  content_age_days
    1 content_cf56e2e2e282                  74.08 meaningful_visibility|stale_visible_page|declining_with_visibility|position_opportunity|low_ctr_visible_page refresh_review            61678          19.7 0.15      5125.0                     194               231
    2 content_6a5b8ccbd700                  67.52                                                                   meaningful_visibility|position_opportunity refresh_review           148534          18.3 0.60      5853.0                     104               153
    3 content_b9f7afeded79     

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd
import numpy as np

# Get the 20 highest-ranked pages from the baseline queue
top20 = output.head(20).copy()

# Add a confidence note and a limitation for each page
def add_confidence_note(row):
    reasons = row["reason_codes"].split("|")

    if (
        "declining_with_visibility" in reasons
        and "low_ctr_visible_page" in reasons
        and "position_opportunity" in reasons
    ):
        confidence = (
            "Multiple review signals are present; "
            "verify the page's search intent and recent performance."
        )
    elif "declining_with_visibility" in reasons:
        confidence = (
            "The page has a decline signal and search visibility; "
            "check whether the decline is sustained."
        )
    elif "low_ctr_visible_page" in reasons:
        confidence = (
            "CTR is low for its visibility and position; "
            "check search intent and the search results page."
        )
    elif "stale_visible_page" in reasons:
        confidence = (
            "The page has not been updated recently; "
            "check whether its information is outdated."
        )
    elif "position_opportunity" in reasons:
        confidence = (
            "The page ranks within the first 20 positions; "
            "check whether a realistic improvement opportunity exists."
        )
    else:
        confidence = (
            "The score suggests review priority; "
            "inspect the page before deciding what to change."
        )

    return confidence


def add_what_could_be_wrong(row):
    reasons = row["reason_codes"].split("|")

    if "declining_with_visibility" in reasons:
        return (
            "The decline label is based on an observed trend and "
            "does not prove the page will continue declining."
        )

    if "low_ctr_visible_page" in reasons:
        return (
            "Low CTR may be normal for the query or affected by "
            "search-result layout and search intent."
        )

    if "stale_visible_page" in reasons:
        return (
            "An old update date does not necessarily mean the "
            "content is outdated."
        )

    if "position_opportunity" in reasons:
        return (
            "A ranking between positions 1 and 20 does not guarantee "
            "that the page can improve."
        )

    return (
        "The score uses fixed heuristic weights and may not reflect "
        "the page's actual business value."
    )


top20["confidence_note"] = top20.apply(
    add_confidence_note, axis=1
)

top20["what_could_make_it_wrong"] = top20.apply(
    add_what_could_be_wrong, axis=1
)

# Keep the review fields easy to inspect
review_columns = [
    "rank",
    "content_id",
    "baseline_action_score",
    "action",
    "reason_codes",
    "confidence_note",
    "what_could_make_it_wrong",
]

top20_review = top20[review_columns].copy()

print("Number of pages reviewed:", len(top20_review))
print("\nTop-20 review:")
display(top20_review)


Number of pages reviewed: 20

Top-20 review:


,rank,content_id,baseline_action_score,action,reason_codes,confidence_note,what_could_make_it_wrong
16751,1,content_cf56e2e2e282,74.08,refresh_review,meaningful_visibility|stale_visible_page|decli...,Multiple review signals are present; verify th...,The decline label is based on an observed tren...
13139,2,content_6a5b8ccbd700,67.52,refresh_review,meaningful_visibility|position_opportunity,The page ranks within the first 20 positions; ...,A ranking between positions 1 and 20 does not ...
20503,3,content_b9f7afeded79,67.22,refresh_review,meaningful_visibility|declining_with_visibilit...,Multiple review signals are present; verify th...,The decline label is based on an observed tren...
14914,4,content_29c0b99ed170,65.81,refresh_review,meaningful_visibility|position_opportunity|low...,CTR is low for its visibility and position; ch...,Low CTR may be normal for the query or affecte...
12170,5,content_eb366e871254,65.65,refresh_review,meaningful_visibility|position_opportunity|low...,CTR is low for its visibility and position; ch...,Low CTR may be normal for the query or affecte...
12045,6,content_c2d929d83eaa,65.25,refresh_review,meaningful_visibility|stale_visible_page|decli...,Multiple review signals are present; verify th...,The decline label is based on an observed tren...
5388,7,content_147ae2b19152,65.19,refresh_review,meaningful_visibility|position_opportunity|low...,CTR is low for its visibility and position; ch...,Low CTR may be normal for the query or affecte...
15751,8,content_65fde7375946,64.93,refresh_review,meaningful_visibility|position_opportunity|low...,CTR is low for its visibility and position; ch...,Low CTR may be normal for the query or affecte...
29680,9,content_4b05c87e3f08,64.84,refresh_review,meaningful_visibility|position_opportunity|low...,CTR is low for its visibility and position; ch...,Low CTR may be normal for the query or affecte...
12435,10,content_003e0d52bd3d,64.75,refresh_review,meaningful_visibility|position_opportunity|low...,CTR is low for its visibility and position; ch...,Low CTR may be normal for the query or affecte...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

After looking at the top 20 pages, I noticed that most of them are being prioritised because they have good search visibility, an opportunity based on their current position, or a low CTR. These are useful starting points, but they do not automatically mean that every page needs a refresh.

For example, a page may have high impressions but still rank poorly. Another page may have a low CTR, but that could be due to the search intent or the way the result appears in Google. I would check these cases manually before recommending any changes.

I also need to be careful about how the score is built. The decline label comes from an already observed trend, so I should not use `trend_direction` or `trend_pct` to calculate the score. I should also avoid using future-period performance, because that information would not be available when deciding which pages to review.

The aim here is to catch recommendations that look convincing in the ranking but may not hold up when the actual page and its data are checked. The final decision to refresh a page should still depend on a human review.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Section 4: Weak picks + leakage check

import pandas as pd

# 1. Check that the ranked queue and top-20 review exist
assert "output" in globals(), "Run Section 2 first: output was not found."
assert "top20" in globals(), "Run Section 3 first: top20 was not found."

print("Ranked queue rows:", len(output))
print("Top-20 rows:", len(top20))

# 2. Check for duplicate content IDs
if "content_id" in output.columns:
    print("Duplicate content IDs:", output["content_id"].duplicated().sum())

# 3. Inspect the top-ranked pages for possible weak picks
check_columns = [
    "rank",
    "content_id",
    "baseline_action_score",
    "reason_codes",
    "action",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

available_columns = [
    col for col in check_columns if col in top20.columns
]

print("\nTop-20 weak-pick review:")
display(top20[available_columns].head(20))

# 4. Count missing values in important review fields
review_fields = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

available_fields = [
    col for col in review_fields if col in top20.columns
]

print("\nMissing values among the top 20:")
print(top20[available_fields].isna().sum())

# 5. Check whether label or trend fields appear in the scored output
possible_leakage_fields = [
    "is_declining",
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "future_performance",
    "future_impressions",
    "future_clicks"
]

found_leakage_fields = [
    col for col in possible_leakage_fields
    if col in output.columns
]

print("\nPotential label/future fields in output:")
print(found_leakage_fields if found_leakage_fields else "None found")

# 6. Reminder: output columns alone cannot prove the scoring code
# is leakage-free. The score formula must also be inspected.
print(
    "\nImportant: inspect the Section 2 score formula to confirm "
    "that trend/label fields and future-period metrics are not used."
)

Ranked queue rows: 30000
Top-20 rows: 20
Duplicate content IDs: 0

Top-20 weak-pick review:


,rank,content_id,baseline_action_score,reason_codes,action,impressions_90d,avg_position,ctr,word_count
16751,1,content_cf56e2e2e282,74.08,meaningful_visibility|stale_visible_page|decli...,refresh_review,61678,19.7,0.15,5125.0
13139,2,content_6a5b8ccbd700,67.52,meaningful_visibility|position_opportunity,refresh_review,148534,18.3,0.60,5853.0
20503,3,content_b9f7afeded79,67.22,meaningful_visibility|declining_with_visibilit...,refresh_review,95333,19.1,0.31,6929.0
14914,4,content_29c0b99ed170,65.81,meaningful_visibility|position_opportunity|low...,refresh_review,40577,20.0,0.41,6660.0
12170,5,content_eb366e871254,65.65,meaningful_visibility|position_opportunity|low...,refresh_review,168060,16.6,0.20,5527.0
12045,6,content_c2d929d83eaa,65.25,meaningful_visibility|stale_visible_page|decli...,refresh_review,7558,17.9,0.20,4758.0
5388,7,content_147ae2b19152,65.19,meaningful_visibility|position_opportunity|low...,refresh_review,37755,19.7,0.30,6517.0
15751,8,content_65fde7375946,64.93,meaningful_visibility|position_opportunity|low...,refresh_review,53398,18.7,0.21,5093.0
29680,9,content_4b05c87e3f08,64.84,meaningful_visibility|position_opportunity|low...,refresh_review,29516,20.0,0.47,5094.0
12435,10,content_003e0d52bd3d,64.75,meaningful_visibility|position_opportunity|low...,refresh_review,37122,19.4,0.18,5960.0



Missing values among the top 20:
impressions_90d    0
avg_position       0
ctr                0
word_count         0
dtype: int64

Potential label/future fields in output:
None found

Important: inspect the Section 2 score formula to confirm that trend/label fields and future-period metrics are not used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.